In [20]:
# Cell 1: Import libraries
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from gensim.models import Word2Vec
import pandas as pd
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [21]:


dataset = pd.read_csv('VKM_dataset_cleaned.csv')

dataset['combined_text'] = (
    dataset['name'].fillna('') + ' ' +
    dataset['shortdescription'].fillna('') + ' ' +
    dataset['description'].fillna('') + ' ' +
    dataset['content'].fillna('') + ' ' +
    dataset['learningoutcomes'].fillna('') + ' ' +
    dataset['module_tags'].fillna('')
)

stop_words = set(stopwords.words('dutch'))
preprocessed_texts = []

for text in dataset['combined_text']:
    text_lower = text.lower()
    text_no_numbers = re.sub(r'\d+', '', text_lower)
    text_no_punct = re.sub(r'[^\w\s]', '', text_no_numbers)
    
    tokens = []
    for sentence in sent_tokenize(text_no_punct):
        words = word_tokenize(sentence)
        filtered = [w for w in words if w not in stop_words]
        tokens.extend(filtered)
    
    preprocessed_texts.append(' '.join(tokens)) 

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preprocessed_texts)
dataset['tfidf_score'] = tfidf_matrix.sum(axis=1).A1

print(tfidf_matrix.shape)


(211, 4101)


In [22]:


scaler = MinMaxScaler()

dataset['interest_match_norm'] = scaler.fit_transform(dataset['interests_match_score'].values.reshape(-1,1))
dataset['popularity_score_norm'] = scaler.fit_transform(dataset['popularity_score'].values.reshape(-1,1))
dataset['tfidf_score_norm'] = scaler.fit_transform(dataset['tfidf_score'].values.reshape(-1,1))
# Bijvoorbeeld 60% interest_match en 40% popularity
dataset['hybrid_score'] = 0.5 * dataset['interest_match_norm'] + 0.1 * dataset['popularity_score_norm'] + 0.4 * dataset['tfidf_score_norm']

top5_hybrid = dataset.sort_values(by='hybrid_score', ascending=False).head(5)
#print(top5_hybrid[['name', 'shortdescription', 'interests_match_score', 'popularity_score', 'hybrid_score']])
print('top 5 keuzemodules:\n', top5_hybrid[['name']])




top 5 keuzemodules:
                                                   name
198                                         stopmotion
182  multdisciplinair samenwerken in een beroepscon...
118                   robotic ai interfaces - optie 2*
51          act for change together nlqf6 30 + 15 ects
112                                    robot challenge
